# **XGBoost Model**

In [1]:
import pandas as pd
# Load and merge all processed data
df_1 = pd.read_csv('../data/hist-data/processed/E0-2024-2025.csv')
df_2 = pd.read_csv('../data/hist-data/processed/E0-2023-2024.csv')
df_3 = pd.read_csv('../data/hist-data/processed/E0-2022-2023.csv')
df_4 = pd.read_csv('../data/hist-data/processed/E0-2021-2022.csv')
df_5 = pd.read_csv('../data/hist-data/processed/E0-2020-2021.csv')

# Merge all dataframes
df = pd.concat([df_1, df_2, df_3, df_4, df_5])

In [2]:
df = df.sort_values(by='Date')

# Choose a split point
split_date = "2024-08-01"
train_df = df[df['Date'] < split_date]
test_df = df[df['Date'] >= split_date]

# Select features and target
features = ['HomePPG_Last5']  # Add more as needed
target = 'HomeWin'

X_train = train_df[features]
y_train = train_df[target]

X_test = test_df[features]
y_test = test_df[target]

In [3]:
# Train XGBoost Model
from xgboost import XGBClassifier

model = XGBClassifier(
    n_estimators=100,
    learning_rate=0.1,
    max_depth=3,
    subsample=0.8,
    colsample_bytree=0.8,
    use_label_encoder=False,
    eval_metric='logloss'  # Required to suppress warning
)

model.fit(X_train, y_train)

/Users/shauryasingh/Desktop/EPL-Betting-Algorithm/betting-algorithm/venv/lib/python3.11/site-packages/xgboost/training.py:183: UserWarning: [23:39:41] WARNING: /Users/runner/work/xgboost/xgboost/src/learner.cc:738: 
Parameters: { "use_label_encoder" } are not used.

  bst.update(dtrain, iteration=i, fobj=obj)


XGBClassifier(base_score=None, booster=None, callbacks=None,
              colsample_bylevel=None, colsample_bynode=None,
              colsample_bytree=0.8, device=None, early_stopping_rounds=None,
              enable_categorical=False, eval_metric='logloss',
              feature_types=None, feature_weights=None, gamma=None,
              grow_policy=None, importance_type=None,
              interaction_constraints=None, learning_rate=0.1, max_bin=None,
              max_cat_threshold=None, max_cat_to_onehot=None,
              max_delta_step=None, max_depth=3, max_leaves=None,
              min_child_weight=None, missing=nan, monotone_constraints=None,
              multi_strategy=None, n_estimators=100, n_jobs=None,
              num_parallel_tree=None, ...)

In [4]:
from sklearn.metrics import accuracy_score, roc_auc_score

y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]  # Probabilities for HomeWin = 1

print("Accuracy:", accuracy_score(y_test, y_pred))
print("AUC:", roc_auc_score(y_test, y_proba))


Accuracy: 0.7491638795986622
AUC: 0.8373572290834804


In [5]:
# Create a DataFrame with the test data and predictions
test_results = pd.DataFrame({
    'Date': test_df['Date'],
    'HomeTeam': test_df['HomeTeam'],
    'AwayTeam': test_df['AwayTeam'],
    'B365H': test_df['B365H'],
    'HomeWin': y_test,
    'HomeWin_Pred': y_pred})

test_results

,Date,HomeTeam,AwayTeam,B365H,HomeWin,HomeWin_Pred
195,2024-08-16,Man United,Fulham,1.60,1,1
105,2024-08-17,Everton,Brighton,2.63,0,0
135,2024-08-17,Ipswich,Liverpool,8.50,0,0
210,2024-08-17,Newcastle,Southampton,1.36,1,1
224,2024-08-17,Nott'm Forest,Bournemouth,2.45,0,0
...,...,...,...,...,...,...
253,2025-04-02,Southampton,Crystal Palace,5.75,0,0
74,2025-04-02,Brighton,Aston Villa,2.05,0,1
223,2025-04-02,Newcastle,Brentford,1.73,1,1
179,2025-04-02,Liverpool,Everton,1.36,1,1


In [6]:
def calculate_profit(row):
    if row['HomeWin_Pred'] == 1:
        if row['HomeWin'] == 1:
            return row['B365H'] - 1  # profit = payout - stake
        else:
            return -1  # lose the stake
    else:
        return 0  # no bet placed

test_results['Profit'] = test_results.apply(calculate_profit, axis=1)

# Total profit and ROI
total_profit = test_results['Profit'].sum()
total_bets = test_results['HomeWin_Pred'].sum()  # count of 1s in PredHomeWin
roi = (total_profit / total_bets) * 100 if total_bets > 0 else 0

print(f"Total profit: ${total_profit:.2f}")
print(f"Total bets placed: {total_bets}")
print(f"ROI: {roi:.2f}%")


Total profit: $33.62
Total bets placed: 120
ROI: 28.02%
